# LipidOS — PDF parsing on A100 (Marker)

Converts the 62 PDFs in `data/pdf/` to clean structured **markdown + tables**,
so we can look at what came out before deciding what to chunk.

**Why Marker, not local text extraction:** PyMuPDF's text layer is glyph-damaged
(`375 cm1`, `υ▷CH2◁`) and can't read tables. Marker is a GPU document-understanding
model (trained on arXiv/PMC papers) that reconstructs prose *and* tables as markdown.
This is the job the A100 is actually for.

**Runtime:** set **Runtime → Change runtime type → A100 GPU** before running.

**Flow:** upload a zip of `data/pdf/` → Marker on the A100 → download `marker_out.zip`.

## 1. Confirm the GPU (this whole notebook is pointless without it)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > A100 GPU.'
print('CUDA device:', torch.cuda.get_device_name(0))

## 2. Install Marker

Pinned so this run is reproducible. First install pulls model weights (~a few min).

In [ ]:
!pip install -q marker-pdf==1.2.3
print('marker installed')

## 3. Upload the PDFs

Locally, zip the folder and keep the zip small enough to upload:

```bash
cd ~/resume_projects/LipidOS && zip -r pdfs.zip data/pdf
```

Then run the cell and pick `pdfs.zip`. (For >100 MB, mount Google Drive instead
and point `PDF_DIR` at the Drive folder.)

In [ ]:
import os, zipfile, glob
from google.colab import files

up = files.upload()                 # choose pdfs.zip
zname = next(iter(up))
os.makedirs('/content/pdfs', exist_ok=True)
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/pdfs')

# Flatten: find every .pdf regardless of nesting, collect into one dir
os.makedirs('/content/in', exist_ok=True)
pdfs = glob.glob('/content/pdfs/**/*.pdf', recursive=True)
for p in pdfs:
    os.rename(p, f'/content/in/{os.path.basename(p)}')
print(f'{len(pdfs)} PDFs ready in /content/in')

## 4. Run Marker on the A100

One markdown file per PDF, tables reconstructed as markdown pipe tables.
`--workers` batches; the A100 has the memory to run several PDFs at once.

In [ ]:
import time
os.makedirs('/content/out', exist_ok=True)
t0 = time.time()
!marker /content/in --output_dir /content/out --output_format markdown --workers 4
print(f'\nMarker finished in {(time.time()-t0)/60:.1f} min')

## 5. Quick look — did tables survive?

Before downloading, sanity-check that pipe tables actually came through, since
the tables are the reason we ran the GPU model at all.

In [ ]:
import glob, re
mds = glob.glob('/content/out/**/*.md', recursive=True)
print(f'{len(mds)} markdown files produced\n')
for m in sorted(mds):
    txt = open(m).read()
    tbl_rows = txt.count('|\n')
    wn = len(re.findall(r'\b(?:1[0-7]\d\d|2[89]\d\d|3[01]\d\d)\b', txt))  # lipid bands
    flag = '  <-- has tables + many bands' if tbl_rows > 10 and wn > 60 else ''
    print(f'  {len(txt)//1000:4}KB  tblrows~{tbl_rows:4}  bands~{wn:4}  {os.path.basename(m)[:40]}{flag}')

## 6. Package for download

Bring `marker_out.zip` back into the repo as `data/marker/`. We'll inspect the
markdown locally and decide, per paper, what to chunk and which tables to trust
(they still need the chemistry-validation gate — Marker can silently drop/repeat
a dense table row).

In [ ]:
import shutil
shutil.make_archive('/content/marker_out', 'zip', '/content/out')
from google.colab import files
files.download('/content/marker_out.zip')